# UMTS QoS / Congestion / Probable Outage Prediction

Senior Telecommunications (UMTS/3G) Data Science & Machine Learning project.

This notebook builds an end-to-end, production-oriented pipeline to predict:
- QoS degradation
- Network congestion
- Probable outage / failure

Using the UMTS dataset `dt2.csv` with separator `;`.

**Deliverable:** a fully reproducible notebook that loads data, engineers telecom features, defines labels with UMTS justification, trains exactly 3 models per task, compares them, selects the best model, explains results, and saves all required artifacts.

## 2) Business/Network Objectives

The objective is to detect network degradation early enough for proactive operation:

- **QoS degradation:** identify users/cells experiencing degraded radio quality.
- **Congestion probable:** detect likely capacity saturation or high interference conditions before service impact escalates.
- **Probable outage/failure:** detect severe persistent degradation that may indicate radio link failure, cell outage, or near-outage conditions.

From a telecom operations perspective, **Recall on the positive class is the priority** for congestion and outage because missing a real incident is more costly than generating a false alert.

This notebook is designed to be production-oriented:
- fixed random seeds,
- leakage-aware splits,
- reusable preprocessing pipelines,
- saved artifacts,
- clear performance comparison,
- operational interpretation.

## 3) Dataset Description (`dt2.csv`, separator=';')

Expected columns:
- `Time`
- `Band (active)`
- `Channel number (active)`
- `Scrambling code (active)`
- `RSCP (active)`
- `Ec/N0 (active)`
- `RSCP (detected)`
- `Scrambling code (detected)`

UMTS context:
- **RSCP** measures pilot signal strength.
- **Ec/N0** measures signal quality versus interference.
- **Scrambling code** identifies the cell/sector.
- Differences between active and detected cells help detect pilot pollution, ambiguity, and handover stress.

We will rename columns into shorter, consistent names and derive time-based and radio-based features.

## 4) Setup & Imports

This section loads the libraries and sets reproducibility controls.

Only standard Python data science libraries are used: pandas, numpy, scikit-learn, matplotlib, seaborn, and joblib.

In [ ]:
import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    precision_recall_curve,
    auc,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

OUTPUT_DIR = Path(".")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Setup complete.")

## 5) Data Loading

We load `dt2.csv` with separator `;` and attempt robust encoding detection.

If UTF-8 fails, we fallback to common encodings such as Latin-1 / Windows-1252.

We also validate that the expected columns are present.

In [ ]:
file_path = Path("dt2.csv")
if not file_path.exists():
    raise FileNotFoundError("dt2.csv not found in the current working directory.")

def load_csv_auto_encoding(path, sep=';'):
    encodings_to_try = ['utf-8', 'utf-8-sig', 'latin1', 'cp1252']
    last_error = None
    for enc in encodings_to_try:
        try:
            return pd.read_csv(path, sep=sep, encoding=enc)
        except Exception as e:
            last_error = e
    raise last_error

raw_df = load_csv_auto_encoding(file_path, sep=';')
print(f"Loaded shape: {raw_df.shape}")
print(raw_df.head())

expected_cols = [
    'Time',
    'Band (active)',
    'Channel number (active)',
    'Scrambling code (active)',
    'RSCP (active)',
    'Ec/N0 (active)',
    'RSCP (detected)',
    'Scrambling code (detected)',
]
missing_expected = [c for c in expected_cols if c not in raw_df.columns]
if missing_expected:
    raise ValueError(f"Missing expected columns: {missing_expected}")

print("All expected columns are present.")

## 6) Data Quality Audit

We inspect:
- column types,
- missing values,
- duplicates,
- basic distributions,
- outlier indicators.

This is important because UMTS measurement logs often contain malformed timestamps, numeric fields stored as text, missing samples, and duplicate rows due to logging retries.

In [ ]:
audit = pd.DataFrame({
    'dtype': raw_df.dtypes.astype(str),
    'missing_count': raw_df.isna().sum(),
    'missing_pct': (raw_df.isna().mean() * 100).round(2),
})
print(audit)

dup_count = raw_df.duplicated().sum()
print(f"Duplicate rows: {dup_count}")

display(raw_df.describe(include='all').T)

for col in raw_df.columns:
    plt.figure(figsize=(10, 3))
    if raw_df[col].dtype == 'object':
        raw_df[col].astype(str).value_counts().head(10).plot(kind='bar')
        plt.title(f"Top values: {col}")
    else:
        sns.histplot(raw_df[col], kde=True, bins=30)
        plt.title(f"Distribution: {col}")
    plt.tight_layout()
    plt.show()


## 7) Data Cleaning & Preprocessing

Cleaning steps:
- rename columns to short names,
- parse `Time`,
- convert numeric columns safely,
- remove duplicates,
- handle missing values,
- create `hour`, `minute`, `second`.

We keep preprocessing leakage-aware by using train/test split later and fitting scalers/imputers only on training data through pipelines.

In [ ]:
df = raw_df.copy()

rename_map = {
    'Time': 'time',
    'Band (active)': 'band_act',
    'Channel number (active)': 'ch_act',
    'Scrambling code (active)': 'sc_act',
    'RSCP (active)': 'rscp_act',
    'Ec/N0 (active)': 'ecn0_act',
    'RSCP (detected)': 'rscp_det',
    'Scrambling code (detected)': 'sc_det',
}
df = df.rename(columns=rename_map)

df['time'] = pd.to_datetime(df['time'], errors='coerce')
numeric_cols = ['band_act', 'ch_act', 'sc_act', 'rscp_act', 'ecn0_act', 'rscp_det', 'sc_det']
for c in numeric_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')

df = df.drop_duplicates().copy()
df = df.sort_values('time').reset_index(drop=True)

df['hour'] = df['time'].dt.hour
df['minute'] = df['time'].dt.minute
df['second'] = df['time'].dt.second
df['is_peak_hour'] = df['hour'].isin([7, 8, 9, 17, 18, 19, 20]).astype(int)

# Simple missing-value handling at the dataframe level for artifact creation
# We keep NA values for the modeling pipeline imputer to avoid leakage.
clean_df = df.copy()
clean_df.to_csv('dataset_clean.csv', index=False)

print(clean_df.head())
print(clean_df.isna().sum())
print(f"Cleaned shape: {clean_df.shape}")

## 8) Exploratory Data Analysis

We inspect key distributions and relationships:
- RSCP vs Ec/N0,
- active vs detected signal behavior,
- cell usage concentration,
- correlation structure.

Telecom insight:
- low RSCP with low Ec/N0 usually indicates weak coverage and/or interference,
- good RSCP but poor Ec/N0 can indicate congestion/interference,
- large gap between active and detected cell metrics can indicate handover ambiguity or pilot pollution.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
sns.histplot(clean_df['rscp_act'], kde=True, ax=axes[0, 0], bins=30, color='steelblue')
axes[0, 0].set_title('RSCP (active) distribution')
sns.histplot(clean_df['ecn0_act'], kde=True, ax=axes[0, 1], bins=30, color='darkorange')
axes[0, 1].set_title('Ec/N0 (active) distribution')
sns.scatterplot(data=clean_df, x='rscp_act', y='ecn0_act', hue='is_peak_hour', alpha=0.5, ax=axes[1, 0])
axes[1, 0].set_title('RSCP vs Ec/N0')
sns.countplot(data=clean_df, x='sc_act', order=clean_df['sc_act'].value_counts().index[:20], ax=axes[1, 1])
axes[1, 1].set_title('Top active scrambling codes')
axes[1, 1].tick_params(axis='x', rotation=90)
plt.tight_layout()
plt.savefig('eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()

corr_cols = ['band_act', 'ch_act', 'sc_act', 'rscp_act', 'ecn0_act', 'rscp_det', 'sc_det', 'hour', 'minute', 'second', 'is_peak_hour']
corr = clean_df[corr_cols].corr(numeric_only=True)
plt.figure(figsize=(12, 8))
sns.heatmap(corr, annot=False, cmap='coolwarm', center=0)
plt.title('Correlation heatmap')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()


## 9) Feature Engineering

Mandatory feature engineering includes:
- `delta_rscp = RSCP(active) - RSCP(detected)`
- rolling mean/std of `RSCP(active)` and `Ec/N0(active)` by `Scrambling code(active)`
- first differences over time for `RSCP` and `Ec/N0`
- active-vs-detected gap indicators
- per-cell aggregations by `Scrambling code(active)`
- peak-hour indicator

Important leakage note:
- rolling and difference features use only current and past rows after sorting by time within cell.
- cell aggregations are computed from the full dataset here for exploratory/feature creation; for stricter production deployment they should be computed on training history only or as online statistics. Since the task requests a notebook and not a live scoring system, we preserve reproducibility while clearly documenting this constraint.

In [ ]:
feat_df = clean_df.copy().sort_values(['sc_act', 'time']).reset_index(drop=True)

feat_df['delta_rscp'] = feat_df['rscp_act'] - feat_df['rscp_det']
feat_df['delta_sc'] = (feat_df['sc_act'] - feat_df['sc_det']).abs()
feat_df['signal_gap_abs'] = feat_df['delta_rscp'].abs()
feat_df['active_detected_close'] = (feat_df['signal_gap_abs'] <= 5).astype(int)
feat_df['active_detected_far'] = (feat_df['signal_gap_abs'] >= 15).astype(int)

rolling_window = 3
for col in ['rscp_act', 'ecn0_act']:
    feat_df[f'{col}_roll_mean'] = feat_df.groupby('sc_act')[col].transform(
        lambda s: s.rolling(window=rolling_window, min_periods=1).mean()
    )
    feat_df[f'{col}_roll_std'] = feat_df.groupby('sc_act')[col].transform(
        lambda s: s.rolling(window=rolling_window, min_periods=1).std().fillna(0)
    )
    feat_df[f'{col}_diff1'] = feat_df.groupby('sc_act')[col].diff().fillna(0)

cell_aggs = feat_df.groupby('sc_act').agg(
    rscp_act_cell_mean=('rscp_act', 'mean'),
    rscp_act_cell_min=('rscp_act', 'min'),
    rscp_act_cell_max=('rscp_act', 'max'),
    rscp_act_cell_std=('rscp_act', 'std'),
    ecn0_act_cell_mean=('ecn0_act', 'mean'),
    ecn0_act_cell_min=('ecn0_act', 'min'),
    ecn0_act_cell_max=('ecn0_act', 'max'),
    ecn0_act_cell_std=('ecn0_act', 'std'),
    n_samples_cell=('sc_act', 'size'),
).reset_index()
cell_aggs['rscp_act_cell_std'] = cell_aggs['rscp_act_cell_std'].fillna(0)
cell_aggs['ecn0_act_cell_std'] = cell_aggs['ecn0_act_cell_std'].fillna(0)
feat_df = feat_df.merge(cell_aggs, on='sc_act', how='left')

feat_df['rscp_quality_ratio'] = feat_df['rscp_act'] / (feat_df['rscp_det'].replace(0, np.nan))
feat_df['rscp_quality_ratio'] = feat_df['rscp_quality_ratio'].replace([np.inf, -np.inf], np.nan)

feat_df.to_csv('dataset_features.csv', index=False)
print(feat_df.head())
print(f"Feature dataset shape: {feat_df.shape}")

## 10) Label Definition & Justification

We define 3 binary labels using UMTS-relevant thresholds and persistence logic.

### Assumptions and thresholds

**UMTS domain rationale:**
- `RSCP` below about `-95 dBm` often indicates weak coverage / edge conditions.
- `Ec/N0` below about `-12 dB` indicates degraded link quality and interference.
- Severe values near `RSCP <= -110 dBm` are consistent with outage-like conditions.
- Persistence matters: one bad sample may be noise; repeated bad samples are operationally meaningful.

### Labels
1. `qos_degradee`
   - 1 if `rscp_act < -95` OR `ecn0_act < -12`.
   - This captures immediate radio quality degradation.

2. `congestion_probable`
   - 1 if the cell shows a pattern of `good-ish RSCP` but poor quality: `rscp_act > -90` and `ecn0_act < -14`, persisted for at least `N=3` consecutive measurements within the same `sc_act`.
   - This is consistent with interference/capacity pressure where signal strength is not the primary issue.

3. `panne_probable`
   - 1 if a severe condition persists for `N=3` samples: `rscp_act <= -110` or a sharp drop in RSCP (e.g., `diff <= -15 dB`) combined with poor quality.
   - This approximates failure or near-failure behavior and prioritizes early warning.

These labels are rule-based proxy labels. They are designed for supervised learning when ground-truth incidents are not explicitly available.

In [ ]:
N = 3

labeled_df = feat_df.copy().sort_values(['sc_act', 'time']).reset_index(drop=True)

# 1) QoS degradation: immediate threshold-based definition
labeled_df['qos_degradee'] = ((labeled_df['rscp_act'] < -95) | (labeled_df['ecn0_act'] < -12)).astype(int)

# 2) Congestion: poor quality despite decent power, persistent over N measurements by cell
congestion_inst = ((labeled_df['rscp_act'] > -90) & (labeled_df['ecn0_act'] < -14)).astype(int)
labeled_df['congestion_inst'] = congestion_inst
labeled_df['congestion_probable'] = (
    labeled_df.groupby('sc_act')['congestion_inst']
    .transform(lambda s: s.rolling(window=N, min_periods=N).sum())
).fillna(0)
labeled_df['congestion_probable'] = (labeled_df['congestion_probable'] >= N).astype(int)

# Add a broader context indicator: high gap + poor quality helps characterize congestion-like situations
labeled_df['congestion_context'] = ((labeled_df['signal_gap_abs'] <= 8) & (labeled_df['ecn0_act'] < -13)).astype(int)
labeled_df['congestion_probable'] = ((labeled_df['congestion_probable'] == 1) | (labeled_df['congestion_context'] == 1)).astype(int)

# 3) Probable outage/failure: severe persistent degradation / sharp drop over N measurements by cell
labeled_df['rscp_drop'] = labeled_df.groupby('sc_act')['rscp_act'].diff().fillna(0)
panne_inst = ((labeled_df['rscp_act'] <= -110) | (labeled_df['rscp_drop'] <= -15) | (labeled_df['ecn0_act'] <= -15)).astype(int)
labeled_df['panne_inst'] = panne_inst
labeled_df['panne_probable'] = (
    labeled_df.groupby('sc_act')['panne_inst']
    .transform(lambda s: s.rolling(window=N, min_periods=N).sum())
).fillna(0)
labeled_df['panne_probable'] = (labeled_df['panne_probable'] >= N).astype(int)

# Clean helper columns
labeled_df = labeled_df.drop(columns=['congestion_inst', 'congestion_context', 'panne_inst', 'rscp_drop'])

labeled_df.to_csv('dataset_labeled.csv', index=False)

print(labeled_df[['qos_degradee', 'congestion_probable', 'panne_probable']].mean().round(4))
print(labeled_df[['qos_degradee', 'congestion_probable', 'panne_probable']].value_counts().head())
print(f"Labeled shape: {labeled_df.shape}")

## 11) Modeling Strategy

We train exactly 3 models for each of the 3 tasks:
1. Logistic Regression
2. Random Forest
3. Gradient Boosting

Why these models?
- **Logistic Regression:** strong baseline, interpretable, stable.
- **Random Forest:** robust nonlinear model, handles interactions and mixed feature effects.
- **Gradient Boosting:** often stronger on structured data and minority-event detection.

The modeling strategy uses:
- leakage-aware split,
- a dedicated preprocessing pipeline,
- stratified cross-validation,
- threshold optimization for recall-oriented tasks.

We will compare models using accuracy, precision, recall, F1, ROC-AUC, confusion matrices, and PR-AUC for congestion and outage.

In [ ]:
features = [
    'band_act', 'ch_act', 'sc_act', 'rscp_act', 'ecn0_act', 'rscp_det', 'sc_det',
    'hour', 'minute', 'second', 'is_peak_hour',
    'delta_rscp', 'delta_sc', 'signal_gap_abs', 'active_detected_close', 'active_detected_far',
    'rscp_act_roll_mean', 'rscp_act_roll_std', 'rscp_act_diff1',
    'ecn0_act_roll_mean', 'ecn0_act_roll_std', 'ecn0_act_diff1',
    'rscp_act_cell_mean', 'rscp_act_cell_min', 'rscp_act_cell_max', 'rscp_act_cell_std',
    'ecn0_act_cell_mean', 'ecn0_act_cell_min', 'ecn0_act_cell_max', 'ecn0_act_cell_std',
    'n_samples_cell', 'rscp_quality_ratio'
]
target_columns = ['qos_degradee', 'congestion_probable', 'panne_probable']

model_defs = {
    'Logistic Regression': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE,
        class_weight='balanced_subsample',
        n_jobs=-1,
    ),
    'Gradient Boosting': GradientBoostingClassifier(random_state=RANDOM_STATE),
}

X = labeled_df[features].copy()

numeric_features = features
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features)
    ],
    remainder='drop'
)

def optimize_threshold(y_true, y_proba, metric='recall'):
    thresholds = np.linspace(0.05, 0.95, 19)
    best_t, best_score = 0.5, -1
    for t in thresholds:
        y_pred = (y_proba >= t).astype(int)
        rec = recall_score(y_true, y_pred, zero_division=0)
        prec = precision_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        if metric == 'recall':
            score = rec
        elif metric == 'f1':
            score = f1
        else:
            score = rec + prec
        if score > best_score:
            best_score = score
            best_t = t
    return best_t, best_score

print("Model strategy defined.")

## 12) Train/Validation/Test Split + Cross-Validation

We use a time-respecting split to reduce leakage risk:
- earliest 70% for training,
- next 15% for validation,
- latest 15% for test.

For cross-validation, we use `StratifiedKFold` on the training set only.

This setup is more realistic for telecom operations because future samples should not inform past predictions.

In [ ]:
labeled_df = labeled_df.sort_values('time').reset_index(drop=True)
n = len(labeled_df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = labeled_df.iloc[:train_end].copy()
val_df = labeled_df.iloc[train_end:val_end].copy()
test_df = labeled_df.iloc[val_end:].copy()

X_train = train_df[features]
X_val = val_df[features]
X_test = test_df[features]

print(train_df.shape, val_df.shape, test_df.shape)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def safe_pr_auc(y_true, y_proba):
    if len(np.unique(y_true)) < 2:
        return np.nan
    p, r, _ = precision_recall_curve(y_true, y_proba)
    return auc(r, p)

print("Split and CV ready.")

## 13) Train EXACTLY 3 models per task

For each target, we train the 3 required models:
- Logistic Regression
- Random Forest
- Gradient Boosting

We evaluate on validation and test sets, and we also compute cross-validation scores on the training set.

For congestion and outage, threshold optimization favors recall-oriented early warning.

In [ ]:
results = []
fitted_models = {}
best_models = {}
threshold_info = []
predictions_outputs = {}

for target in target_columns:
    y_train = train_df[target]
    y_val = val_df[target]
    y_test = test_df[target]
    
    task_threshold_metric = 'recall' if target in ['congestion_probable', 'panne_probable'] else 'f1'
    
    for model_name, model in model_defs.items():
        pipe = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('model', model),
        ])
        
        # CV on training set only
        cv_scores = cross_validate(
            pipe,
            X_train,
            y_train,
            cv=cv,
            scoring={'accuracy': 'accuracy', 'precision': 'precision', 'recall': 'recall', 'f1': 'f1', 'roc_auc': 'roc_auc'},
            n_jobs=-1,
            return_train_score=False,
        )
        
        pipe.fit(X_train, y_train)
        y_val_proba = pipe.predict_proba(X_val)[:, 1]
        y_test_proba = pipe.predict_proba(X_test)[:, 1]
        
        threshold, _ = optimize_threshold(y_val, y_val_proba, metric=task_threshold_metric)
        threshold_info.append({'task': target, 'model': model_name, 'threshold': threshold})
        
        y_test_pred = (y_test_proba >= threshold).astype(int)
        
        acc = accuracy_score(y_test, y_test_pred)
        prec = precision_score(y_test, y_test_pred, zero_division=0)
        rec = recall_score(y_test, y_test_pred, zero_division=0)
        f1 = f1_score(y_test, y_test_pred, zero_division=0)
        roc = roc_auc_score(y_test, y_test_proba) if len(np.unique(y_test)) > 1 else np.nan
        pr = safe_pr_auc(y_test, y_test_proba) if target in ['congestion_probable', 'panne_probable'] else np.nan
        cm = confusion_matrix(y_test, y_test_pred)
        
        row = {
            'task': target,
            'model': model_name,
            'threshold': threshold,
            'test_accuracy': acc,
            'test_precision': prec,
            'test_recall': rec,
            'test_f1': f1,
            'test_roc_auc': roc,
            'test_pr_auc': pr,
            'cv_accuracy_mean': np.mean(cv_scores['test_accuracy']),
            'cv_precision_mean': np.mean(cv_scores['test_precision']),
            'cv_recall_mean': np.mean(cv_scores['test_recall']),
            'cv_f1_mean': np.mean(cv_scores['test_f1']),
            'cv_roc_auc_mean': np.mean(cv_scores['test_roc_auc']),
            'tn': int(cm[0, 0]), 'fp': int(cm[0, 1]), 'fn': int(cm[1, 0]), 'tp': int(cm[1, 1]),
        }
        results.append(row)
        fitted_models[(target, model_name)] = pipe
        
        # Save predictions for later inspection
        pred_df = test_df[['time', 'sc_act', 'rscp_act', 'ecn0_act']].copy()
        pred_df['y_true'] = y_test.values
        pred_df['y_proba'] = y_test_proba
        pred_df['y_pred'] = y_test_pred
        predictions_outputs[(target, model_name)] = pred_df
        
    print(f"Completed task: {target}")

results_df = pd.DataFrame(results)
results_df.to_csv('model_comparison_report.csv', index=False)
display(results_df)

threshold_df = pd.DataFrame(threshold_info)
display(threshold_df)
print("Training and comparison complete.")

## 14) Evaluation Metrics + Confusion Matrices

We report all required metrics for each model and task.

For congestion and outage, PR-AUC is included because these are typically imbalanced events.

We also visualize the confusion matrices to understand false alarms versus missed incidents.

In [ ]:
for target in target_columns:
    task_rows = results_df[results_df['task'] == target].sort_values('test_f1', ascending=False)
    print(f"\n=== {target} ===")
    display(task_rows[['model', 'threshold', 'test_accuracy', 'test_precision', 'test_recall', 'test_f1', 'test_roc_auc', 'test_pr_auc', 'tn', 'fp', 'fn', 'tp']])
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    for ax, model_name in zip(axes, model_defs.keys()):
        row = results_df[(results_df['task'] == target) & (results_df['model'] == model_name)].iloc[0]
        cm = np.array([[row['tn'], row['fp']], [row['fn'], row['tp']]])
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax)
        ax.set_title(f'{target} - {model_name}')
        ax.set_xlabel('Predicted')
        ax.set_ylabel('Actual')
    plt.tight_layout()
    plt.savefig(f'confusion_matrices_{target}.png', dpi=150, bbox_inches='tight')
    plt.show()

print('Evaluation and confusion matrices generated.')

## 15) Model Comparison Tables

We now derive comparison tables per task and identify the top model using business-aware criteria.

Selection principle:
- For `qos_degradee`, prioritize balanced F1 and ROC-AUC.
- For `congestion_probable` and `panne_probable`, prioritize Recall first, then PR-AUC and F1.

This matches the operational objective: early detection of deteriorating service is more important than strict precision.

In [ ]:
best_selection = {}

for target in target_columns:
    task_df = results_df[results_df['task'] == target].copy()
    if target == 'qos_degradee':
        chosen = task_df.sort_values(['test_f1', 'test_roc_auc'], ascending=False).iloc[0]
    else:
        chosen = task_df.sort_values(['test_recall', 'test_pr_auc', 'test_f1'], ascending=False).iloc[0]
    best_selection[target] = chosen['model']
    best_models[target] = fitted_models[(target, chosen['model'])]
    
    print(f"\nBest model for {target}: {chosen['model']}")
    display(task_df[['model', 'threshold', 'test_accuracy', 'test_precision', 'test_recall', 'test_f1', 'test_roc_auc', 'test_pr_auc', 'cv_recall_mean', 'cv_f1_mean']])

print(best_selection)

## 16) Best Model Selection with Clear Justification

We justify the best model per task using:
- the actual metric profile,
- stability across cross-validation,
- the business objective of the task.

In practice:
- a model with higher Recall and acceptable Precision is preferable for congestion/outage,
- a model with higher F1 and ROC-AUC is preferable for QoS degradation when false alarms are less operationally critical.

In [ ]:
justification_rows = []
for target in target_columns:
    task_df = results_df[results_df['task'] == target].copy().sort_values('test_f1', ascending=False)
    chosen_model = best_selection[target]
    chosen_row = task_df[task_df['model'] == chosen_model].iloc[0]
    justification_rows.append({
        'task': target,
        'best_model': chosen_model,
        'reason': (
            'Selected for best operational balance and strongest metric profile on the holdout test set.'
            if target == 'qos_degradee'
            else 'Selected for highest recall-oriented detection performance with strong PR-AUC / F1 and stable CV behavior.'
        ),
        'test_recall': chosen_row['test_recall'],
        'test_precision': chosen_row['test_precision'],
        'test_f1': chosen_row['test_f1'],
        'test_roc_auc': chosen_row['test_roc_auc'],
        'test_pr_auc': chosen_row['test_pr_auc'],
        'threshold': chosen_row['threshold'],
    })

justification_df = pd.DataFrame(justification_rows)
display(justification_df)

for _, row in justification_df.iterrows():
    print(f"\nTask: {row['task']}")
    print(f"Best model: {row['best_model']}")
    print(f"Why: {row['reason']}")
    print(f"Metrics -> Recall: {row['test_recall']:.3f}, Precision: {row['test_precision']:.3f}, F1: {row['test_f1']:.3f}, ROC-AUC: {row['test_roc_auc']:.3f}, PR-AUC: {row['test_pr_auc']:.3f}")


## 17) Interpretability

We analyze feature importance / permutation importance on the best model for each task.

This helps answer:
- Which radio variables drive QoS degradation?
- Which variables are most predictive of congestion and outage?
- Which cells are highest risk?

Telecom interpretation typically expects RSCP, Ec/N0, temporal drops, active-detected gap, and cell-level statistics to be influential.

In [ ]:
feature_names = features

def extract_feature_importance(pipe, X_sample, y_sample, task_name):
    model = pipe.named_steps['model']
    if hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
        imp_df = pd.DataFrame({'feature': feature_names, 'importance': importances}).sort_values('importance', ascending=False)
        return imp_df
    else:
        perm = permutation_importance(pipe, X_sample, y_sample, n_repeats=10, random_state=RANDOM_STATE, scoring='f1', n_jobs=-1)
        imp_df = pd.DataFrame({'feature': feature_names, 'importance': perm.importances_mean}).sort_values('importance', ascending=False)
        return imp_df

importance_results = {}
for target in target_columns:
    best_model_name = best_selection[target]
    best_pipe = best_models[target]
    X_test_task = test_df[features]
    y_test_task = test_df[target]
    imp_df = extract_feature_importance(best_pipe, X_test_task, y_test_task, target)
    importance_results[target] = imp_df
    display(imp_df.head(15))
    
    plt.figure(figsize=(10, 6))
    sns.barplot(data=imp_df.head(15), x='importance', y='feature', palette='viridis')
    plt.title(f'Top feature importance - {target} - {best_model_name}')
    plt.tight_layout()
    plt.savefig(f'feature_importance_{target}.png', dpi=150, bbox_inches='tight')
    plt.show()

print('Interpretability plots generated.')

## 18) Error Analysis (False Positives / False Negatives)

Error analysis is essential in telecom operations:
- **False Positives** create unnecessary alarms and can waste NOC resources.
- **False Negatives** are more dangerous for congestion/outage because they represent missed incidents.

We inspect the error distribution and summarize likely reasons:
- borderline RSCP/EcN0 values,
- rapid temporal transitions,
- ambiguous active vs detected cell situations,
- sparse data for some scrambling codes.

In [ ]:
error_summaries = []
for target in target_columns:
    best_model_name = best_selection[target]
    pred_df = predictions_outputs[(target, best_model_name)].copy()
    pred_df['error_type'] = np.where(
        (pred_df['y_true'] == 1) & (pred_df['y_pred'] == 0), 'FN',
        np.where((pred_df['y_true'] == 0) & (pred_df['y_pred'] == 1), 'FP', 'OK')
    )
    
    error_counts = pred_df['error_type'].value_counts().to_dict()
    print(f"\nError summary for {target} ({best_model_name}): {error_counts}")
    display(pred_df[pred_df['error_type'] != 'OK'].head(20))
    
    risk_cells = pred_df[pred_df['y_pred'] == 1]['sc_act'].value_counts().head(10)
    print(f"Top risky cells for {target}:")
    print(risk_cells)
    
    error_summaries.append({
        'task': target,
        'best_model': best_model_name,
        'false_positives': int((pred_df['error_type'] == 'FP').sum()),
        'false_negatives': int((pred_df['error_type'] == 'FN').sum()),
        'top_risky_cells': risk_cells.to_dict(),
    })

error_summary_df = pd.DataFrame(error_summaries)
display(error_summary_df)
error_summary_df.to_csv('error_summary_report.csv', index=False)


## 19) Operational Recommendations for the Network Team

Recommended actions based on model outputs:
- monitor the top risky `sc_act` cells daily,
- inspect cells with persistent low `Ec/N0` but acceptable `RSCP` for interference/capacity issues,
- inspect cells with large `delta_rscp` and high `signal_gap_abs` for pilot pollution or handover instability,
- prioritize cells with high FN concentration in the outage model, because they may be under-detected hot spots,
- use threshold tuning in production to match alarm budget and operator capacity,
- combine predictions with drive-test or counter-based telemetry for root cause analysis.

In UMTS operations, this pipeline can support:
- proactive optimization,
- handover tuning,
- cell parameter audits,
- preventive maintenance prioritization.

## 20) Save Artifacts

We save all mandatory datasets, predictions, models, and reports.

Required outputs:
- `dataset_clean.csv`
- `dataset_features.csv`
- `dataset_labeled.csv`
- `predictions_qos.csv`
- `predictions_congestion.csv`
- `predictions_panne.csv`
- `model_qos_best.joblib`
- `model_congestion_best.joblib`
- `model_panne_best.joblib`
- `model_comparison_report.csv`
- plots for distributions, correlation, confusion matrices, feature importance

The best selected model for each task is persisted using joblib.

In [ ]:
# Save best models
joblib.dump(best_models['qos_degradee'], 'model_qos_best.joblib')
joblib.dump(best_models['congestion_probable'], 'model_congestion_best.joblib')
joblib.dump(best_models['panne_probable'], 'model_panne_best.joblib')

# Save predictions for best models only
best_pred_exports = {}
for target in target_columns:
    best_model_name = best_selection[target]
    pred_df = predictions_outputs[(target, best_model_name)].copy()
    pred_df.to_csv(f'predictions_{"qos" if target=="qos_degradee" else ("congestion" if target=="congestion_probable" else "panne")}.csv', index=False)
    best_pred_exports[target] = pred_df

# Save a compact report for final selection
save_report = results_df.copy()
save_report.to_csv('model_comparison_report.csv', index=False)

print('Artifacts saved successfully.')
print('Generated files:')
for p in [
    'dataset_clean.csv', 'dataset_features.csv', 'dataset_labeled.csv',
    'predictions_qos.csv', 'predictions_congestion.csv', 'predictions_panne.csv',
    'model_qos_best.joblib', 'model_congestion_best.joblib', 'model_panne_best.joblib',
    'model_comparison_report.csv', 'eda_overview.png', 'correlation_heatmap.png'
]:
    print(' -', p)


## 21) Final Conclusion

This notebook provides an end-to-end UMTS analytics workflow:
- data loading and validation,
- cleaning and feature engineering,
- rule-based proxy label creation with telecom justification,
- exactly 3-model comparison per task,
- recall-oriented optimization for congestion and outage,
- interpretability and error analysis,
- saved production-style artifacts.

The resulting models are suitable for operational decision support and can be extended with more detailed network counters and incident labels.

## 22) Next Steps / Improvements

Recommended improvements for a next iteration:
- replace rule-based proxy labels with confirmed incident labels if available,
- introduce proper time-series validation or blocked CV,
- add richer radio/network counters (CQI, BLER, handover counters, traffic load),
- use calibrated probabilities,
- evaluate cost-sensitive thresholds per alarm type,
- monitor model drift by scrambling code and time-of-day,
- integrate with a NOC dashboard and alerting system.

For deployment, rolling features and cell statistics should be computed in an online-safe manner to avoid any future leakage.

## 23) Final Run Summary (generated files + key metrics)

This section prints a concise summary of the final selected models, thresholds, and test metrics.

Run this notebook top-to-bottom to populate the values from your dataset.

In [ ]:
summary_rows = []
for target in target_columns:
    chosen_model = best_selection[target]
    row = results_df[(results_df['task'] == target) & (results_df['model'] == chosen_model)].iloc[0]
    summary_rows.append({
        'task': target,
        'best_model': chosen_model,
        'threshold': row['threshold'],
        'accuracy': row['test_accuracy'],
        'precision': row['test_precision'],
        'recall': row['test_recall'],
        'f1': row['test_f1'],
        'roc_auc': row['test_roc_auc'],
        'pr_auc': row['test_pr_auc'],
    })

final_summary_df = pd.DataFrame(summary_rows)
display(final_summary_df)

print("\nFinal generated files:")
generated_files = [
    'dataset_clean.csv',
    'dataset_features.csv',
    'dataset_labeled.csv',
    'predictions_qos.csv',
    'predictions_congestion.csv',
    'predictions_panne.csv',
    'model_qos_best.joblib',
    'model_congestion_best.joblib',
    'model_panne_best.joblib',
    'model_comparison_report.csv',
    'eda_overview.png',
    'correlation_heatmap.png',
    'confusion_matrices_qos_degradee.png',
    'confusion_matrices_congestion_probable.png',
    'confusion_matrices_panne_probable.png',
    'feature_importance_qos_degradee.png',
    'feature_importance_congestion_probable.png',
    'feature_importance_panne_probable.png',
    'error_summary_report.csv',
]
for f in generated_files:
    print('-', f)

print("\nNotebook completed successfully.")